# Classifier data prep
Steps: load, clean, map trades, parse prices, one-row-per-service table, group split.
Run top to bottom.

In [ ]:
import pandas as pd, re
from google.colab import files
uploaded = files.upload()
raw = pd.read_csv(list(uploaded.keys())[0])
print(raw.shape)

## 1. Clean and map trades
Fix the `ac_service_repairs` typo and map the four source categories to the three app trades (AC + microwave -> appliance). Normalise names so case-only variants collapse (e.g. '3 Phase Changeover Switch Installation' vs '3 Phase changeover switch installation').

In [ ]:
TRADE_MAP = {'electricians': 'electrician', 'plumbers': 'plumber',
             'ac_service_repair': 'appliance', 'ac_service_repairs': 'appliance',
             'microwave_repair': 'appliance'}
df = raw.copy()
df['trade'] = df['service'].map(TRADE_MAP)
assert df['trade'].notna().all()
df['name'] = df['subservice_name'].str.strip().str.replace(r'\s+', ' ', regex=True)
df['name_key'] = df['name'].str.lower()
print(df['trade'].value_counts())
print('raw unique names:', df['subservice_name'].nunique(), '| after normalising:', df.groupby('trade')['name_key'].nunique().sum())

## 2. Parse prices
Each charge is a list like `['₹900', '₹499 (Labour)']`. `total_price` = sum of all amounts; `has_labour_split` flags the two-part rows. (No thousands separators exist in this file, so the commas are just list separators.)

In [ ]:
def parse_total(s):
    nums = [int(n) for n in re.findall(r'₹(\d+)', s)]
    return pd.Series({'total_price': sum(nums), 'has_labour_split': len(nums) > 1})
df[['total_price', 'has_labour_split']] = df['subservice_charge'].apply(parse_total)
print(df['total_price'].describe())
print(df['has_labour_split'].value_counts())
print('zero or missing prices:', (df['total_price'] <= 0).sum())

## 3. Remove exact duplicates and build one row per service
The same service repeats in about 40 cities, so the classifier works on unique services, not 8,000 rows. Prices are kept for later (median across cities).

In [ ]:
print('exact duplicate rows:', df.duplicated(['trade', 'name_key', 'city_name', 'subservice_charge']).sum())
df = df.drop_duplicates(['trade', 'name_key', 'city_name', 'subservice_charge'])

# a name used under two trades would be an ambiguous label
amb = df.groupby('name_key')['trade'].nunique()
ambiguous = amb[amb > 1].index.tolist()
print('names under more than one trade:', ambiguous)
# 'Material Procurement Charges' is a billing line, not a customer problem - drop it from the classifier data
df = df[~df['name_key'].isin(ambiguous)]

services = (df.groupby(['trade', 'name_key'])
              .agg(name=('name', 'first'), n_cities=('city_name', 'nunique'),
                   median_price_inr=('total_price', 'median'),
                   source=('service', lambda x: '|'.join(sorted(set(x)))))
              .reset_index())
print(len(services), 'unique services')
print(services['trade'].value_counts())
services.head(10)

## 4. Group split by service
Split the unique services (not rows) 70/15/15, stratified by trade, so no service appears in two sets. This is the right split for the **trade-level classifier** (can it recognise a service it has never seen?) and for the **price model**. For the service-level classifier the split is on customer phrasings instead - see the notes in the chat.

In [ ]:
from sklearn.model_selection import train_test_split
train, rest = train_test_split(services, test_size=0.30, stratify=services['trade'], random_state=42)
val, test = train_test_split(rest, test_size=0.50, stratify=rest['trade'], random_state=42)
services['split'] = 'train'
services.loc[val.index, 'split'] = 'val'
services.loc[test.index, 'split'] = 'test'
print(pd.crosstab(services['trade'], services['split']))
services.to_csv('services_split.csv', index=False)
files.download('services_split.csv')